In [ ]:
from google.colab import drive
drive.mount('/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset/yolo')

In [1]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.0/107.7 GB disk)


In [10]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt") # 'yolov8s-seg.pt' yerine 'yolov8s.pt' kullanıldı

In [11]:
# Eğitimi Başlat
# Önceki hataya göre, datasetinizde segmentasyon etiketleri yerine sınırlayıcı kutu (tespit) etiketleri bulunmaktadır.
# Bu nedenle, 'yolov8s-seg.pt' segmentasyon modeli yerine, tespit için 'yolov8s.pt' modelini kullanmanız gerekmektedir.
# Model seçimini değiştirmek için lütfen 'cFi_uJsHygrJ' ID'li hücreyi düzenleyin.
# Oradaki 'model = YOLO("yolov8s-seg.pt")' satırını 'model = YOLO("yolov8s.pt")' olarak değiştirin.
results = model.train(
    data='/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset/yolo/deforestation/data.yaml',
    epochs=50,                  # Eğitim tur sayısı
    imgsz=512,                  # İndirdiğimiz görsel boyutu
    batch=16,                   # GPU hafızasına göre ayarlanır
    project='/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset', # Sonuçların kaydedileceği yer
    name='marmara_deforestation_v1'
)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset/yolo/deforestation/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.9

In [24]:
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os # Dizin oluşturmak için eklendi

# Eğitilen en iyi modeli yükle
# Bu yol, daha önce belirlenen proje klasörünüzdeki 'best.pt' modelini işaret etmelidir.
# Hata düzeltildi: Modelin doğru yolu, eğitim çıktısında belirtilen yer olmalıdır.
# Eğitim çıktısı: '/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset/marmara_deforestation_v1-2/weights/best.pt'
best_model = YOLO('/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset/marmara_deforestation_v1-2/weights/best.pt') # Modelin doğru yolu

# Tahmin yapılacak görsellerin yolları
# Düzeltildi: Görsellerin orijinal konumuna geri dönüldü.
# Bulunan gerçek dosya isimleri ve yolları ile güncellendi.
img_2010 = '/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset/yolo/deforestation/images/val/canakkale_kazdaglari_2015.png'
img_2025 = '/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset/yolo/deforestation/images/val/canakkale_kazdaglari_2025.png'

# Modelleri görseller üzerinde çalıştır
# 'verbose=False' ekleyerek gereksiz çıktıları engelledik, sadece sonuçlar önemli olduğunda kullanılabilir.
res_2010 = best_model(img_2010, verbose=False)
res_2025 = best_model(img_2025, verbose=False)

# Tahmin sonuçlarını kaydetmek için bir dizin oluştur
output_dir = '/content/drive/MyDrive/Asusx515j/UlutekEkip/Data/Geomorphosis_dataset/inference_results'
os.makedirs(output_dir, exist_ok=True) # Eğer dizin yoksa oluştur, varsa hata verme

# Tahmin sonuçlarını kaydet
# Kaydedilen dosyaların isimlerini ve dizinlerini daha düzenli hale getirdik.
# 'save_txt=True' ile etiket dosyalarını da kaydedebilirsiniz (opsiyonel).
res_2010[0].save(filename=os.path.join(output_dir, 'result_2015.jpg')) # 2010 yerine 2015 olarak güncellendi
res_2025[0].save(filename=os.path.join(output_dir, 'result_2025.jpg'))

# Tespit Edilen Nesne Sayılarını Karşılaştır
# 'boxes' objesinin varlığını kontrol etmek iyi bir pratiktir, bazen tespit olmayabilir.
count_2010 = len(res_2010[0].boxes) if res_2010[0].boxes is not None else 0
count_2025 = len(res_2025[0].boxes) if res_2025[0].boxes is not None else 0

print(f"📊 2015 Yılında Tespit Edilen Bozulma Bölgesi Sayısı: {count_2010}") # 2010 yerine 2015 olarak güncellendi
print(f"📊 2025 Yılında Tespit Edilen Bozulma Bölgesi Sayısı: {count_2025}")

# Yüzde değişimi hesapla
# Paydada sıfır bölme hatasını önlemek için 'count_2010' değeri sıfırsa '1e-5' ekledik.
# Bu, başlangıçta hiç tespit yoksa bile mantıklı bir yüzde değişimi hesaplamamızı sağlar.
if count_2010 == 0 and count_2025 == 0:
    print("⚠️ Her iki yılda da bozulma bölgesi tespit edilemedi.")
elif count_2010 == 0 and count_2025 > 0:
    print(f"⚠️ 2015'te tespit yokken, 2025'te {count_2025} bozulma bölgesi tespit edildi (Büyük Artış). ") # 2010 yerine 2015 olarak güncellendi
else:
    percentage_change = ((count_2025 - count_2010) / count_2010) * 100
    print(f"⚠️ Yüzde Değişim: %{percentage_change:.2f}")

📊 2015 Yılında Tespit Edilen Bozulma Bölgesi Sayısı: 24
📊 2025 Yılında Tespit Edilen Bozulma Bölgesi Sayısı: 17
⚠️ Yüzde Değişim: %-29.17
